In [3]:
from typing import List, Dict, Any, Optional, Tuple
from app.vector_db import get_qa_chain

def _parse_lineage_page_content(page_content: str) -> Dict[str, Optional[str]]:
    """
    Parse a Document.page_content that was produced by _lineage_row_to_document/_row_to_document
    into a structured lineage dict. Unknown keys are ignored.
    """
    expected_keys = {
        "source_database",
        "source_schema",
        "source_table",
        "source_column",
        "target_database",
        "target_schema",
        "target_table",
        "target_column",
        "query_id",
        "query_type",
        "dbt_model_file_path",
        "dependency_score",
    }
    parsed: Dict[str, Optional[str]] = {k: None for k in expected_keys}
    for line in page_content.splitlines():
        if ": " not in line:
            continue
        key, val = line.split(": ", 1)
        key = key.strip()
        if key in parsed:
            parsed[key] = val.strip()
    return parsed

def extract_lineage(
    org_id: str,
    question: str,
    k: int = 8,
) -> Dict[str, Any]:
    """
    Answer a lineage question using the org's vector store and return:
      - answer: LLM synthesized answer
      - lineage_entries: parsed structured lineage rows from retrieved docs
      - sources: minimal source attribution for transparency
    """
    qa = get_qa_chain(org_id=org_id, k=k)
    # For modern LangChain, invoke with a dict to ensure sources are returned
    result = qa.invoke({"query": question})
    answer: str = result.get("result", "") or result.get("answer", "")

    source_docs = result.get("source_documents", []) or []
    lineage_entries: List[Dict[str, Optional[str]]] = []
    sources_min: List[Dict[str, Any]] = []

    for doc in source_docs:
        # Parse the page_content into structured lineage fields when possible
        parsed = _parse_lineage_page_content(getattr(doc, "page_content", "") or "")
        lineage_entries.append(parsed)

        # Minimal source metadata for attribution
        sources_min.append({
            "metadata": getattr(doc, "metadata", {}) or {},
            "preview": (getattr(doc, "page_content", "") or "").splitlines()[:6],
        })

    return {
        "answer": answer,
        "lineage_entries": lineage_entries,
        "sources": sources_min,
    }

E0000 00:00:1762424468.902905 7628451 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762424468.906951 7628451 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [4]:
extract_lineage(org_id='76d33fb3-6062-456b-a211-4aec9971f8be',
question='can you give me the lineage for gl_summary table')

/Users/deepkatbamna/Desktop/query_guard/QueryGuard-Backend/app/vector_db.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


{'answer': 'Here is the lineage information for the `gl_summary` table:\n\n*   **source_database**: `transform_zone`\n*   **source_schema**: `edw`\n*   **source_table**: `gl_summary`\n*   **source_column**: `accountcd`\n    *   **target_database**: `transform_zone`\n    *   **target_schema**: `edw`\n    *   **target_table**: `allocationrule`\n    *   **target_column**: `allocationrulekey`\n*   **source_database**: `transform_zone`\n*   **source_schema**: `edw`\n*   **source_table**: `gl_summary`\n*   **source_column**: `accountcd`\n    *   **target_database**: `transform_zone`\n    *   **target_schema**: `edw`\n    *   **target_table**: `allocationrule`\n    *   **target_column**: `accountfrom`\n*   **source_database**: `transform_zone`\n*   **source_schema**: `edw`\n*   **source_table**: `gl_summary`\n*   **source_column**: `accountcd`\n    *   **target_database**: `transform_zone`\n    *   **target_schema**: `edw`\n    *   **target_table**: `allocationrule`\n    *   **target_column**

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


def get_lineage(query: str) -> str:
    """Get lineage for a given query."""

    
    return f"The lineage for the query {query} is {query}"

# Wrap your function as a LangChain Tool
weather_tool = Tool(
    name="get_weather",
    func=get_weather,
    description="Get the weather for a given city."
)

# Choose GPT-4 Omni model
llm = ChatOpenAI(
    model="gpt-4.1-mini",  # or gpt-4.1, gpt-4.1-pro
    temperature=0
)

# Create the Agent
agent = initialize_agent(
    tools=[weather_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run
result = agent.invoke("what is the weather in sf")
print(result)


/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_26242/996246303.py:25: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(




> Entering new AgentExecutor chain...
Thought: The question asks for the weather in "sf," which likely refers to San Francisco. I should get the weather for San Francisco.
Action: get_weather
Action Input: San Francisco
Observation: It's always sunny in San Francisco!
Thought:Thought: I now know the final answer
Final Answer: The weather in San Francisco is sunny.

> Finished chain.
{'input': 'what is the weather in sf', 'output': 'The weather in San Francisco is sunny.'}


# Tool 1 Vector QA (column lineage)

In [4]:
from typing import Dict, Any
from langchain.agents import Tool, initialize_agent, AgentType
from app.vector_db import get_qa_chain, LLM

# Configure the org/collection to search
ORG_ID = "76d33fb3-6062-456b-a211-4aec9971f8be"  # <-- change to your org collection name

def get_lineage_answer(question: str) -> str:
    """
    Simple tool function: queries the org's lineage vector store and returns an answer.
    Includes a short source preview for transparency.
    """
    qa = get_qa_chain(org_id=ORG_ID, k=8)
    result: Dict[str, Any] = qa.invoke({"query": question})
    answer = result.get("result", "") or result.get("answer", "")

    # Minimal source preview (first few lines of each Document)
    previews = []
    for doc in result.get("source_documents", []) or []:
        preview = (getattr(doc, "page_content", "") or "").splitlines()[:5]
        previews.append("\n".join(preview))

    sources_block = "\n\nSources:\n- " + "\n- ".join(p for p in previews if p) if previews else ""
    return (answer or "No answer found.") + sources_block

# Wrap as a Tool
extract_lineage_tool = Tool(
    name="extract_lineage",
    func=get_lineage_answer,
    description="Answer lineage questions using the org's vector store. Input is a natural-language question."
)

# Use your existing Gemini LLM from app/vector_db.py
agent = initialize_agent(
    tools=[extract_lineage_tool],
    llm=LLM,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Example run
resp = agent.invoke("Which are the tables and columns feeds fundlist in dimsharedservicesallocationrule?")
print(resp)



> Entering new AgentExecutor chain...
I need to find the tables and columns that feed into the fundlist column of the dimsharedservicesallocationrule table.
Action: extract_lineage
Action Input: Which tables and columns feed fundlist in dimsharedservicesallocationrule?
Observation: Based on the information provided, I cannot determine the tables and columns that feed fundlist in dimsharedservicesallocationrule. To provide that information, I would need access to the metadata of the data warehouse, including data lineage information.
Thought:I don't have the information to answer the question.
Final Answer: I am unable to answer this question.

> Finished chain.
{'input': 'Which are the tables and columns feeds fundlist in dimsharedservicesallocationrule?', 'output': 'I am unable to answer this question.'}


# Tool 2

In [3]:
from langchain.agents import initialize_agent, AgentType
from app.tools import build_org_lineage_tool, build_org_query_history_tool, LLM

org_id = "76d33fb3-6062-456b-a211-4aec9971f8be"

# Create both tools
lineage_tool = build_org_lineage_tool(org_id=org_id, k=8)
query_history_tool = build_org_query_history_tool(org_id=org_id, max_iters=5)

# Create agent with both tools
agent = initialize_agent(
    tools=[lineage_tool, query_history_tool],
    llm=LLM,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

# The agent can now answer both types of questions:
# - Lineage questions: "Which tables feed customer.email?"
# - Impact questions: "I am going to remove column customer.email, what queries are impacted?"
resp = agent.invoke("I am going to remove column gl_summary.deptcd, what queries are impacted?")
print(resp)



> Entering new AgentExecutor chain...
The user is asking about the impact of removing a column, so I should use the `query_history_search` tool to find potentially impacted queries.
Action: query_history_search
Action Input: I am going to remove column gl_summary.deptcd

2025-11-26 17:02:14,035 [INFO] lineage.retrieval - org=76d33fb3-6062-456b-a211-4aec9971f8be query_len=45 docs=0



Observation: 📊 Impact Analysis Report:
📌 **Change Summary**

Change summary description: _Removing a column can break downstream processes that rely on it, including views, ETL jobs, dashboards, and queries. SELECT * statements are particularly vulnerable._

| Field                  | Description |
|------------------------|-------------|
| Change Type            | Drop Column |
| Affected Table         | gl_summary |
| Affected Column(s)     | deptcd |
| Requested Change       | Remove column deptcd |
| Reason for Change      | N/A |

---

### **Downstream Impact Analysis**

_No downstream dependencies were found in the provided lineage snippets._

**Explanation:**
No downstream dependencies were found in the provided lineage snippets. Therefore, no impact analysis can be performed.

⚠️ No impacted queries found in query history.
Thought:I have analyzed the impact of removing the column gl_summary.deptcd and found no impacted queries in the query history.

Final Answer: No impacted q

# Tool 3 : PR/Repo Analysis

In [1]:
from app.tools.pr_repo import fetch_pr_analyses_for_org
rows = fetch_pr_analyses_for_org(
    "76d33fb3-6062-456b-a211-4aec9971f8be",
    repo_full_name="Intellytics-Solutions/github-app-poc",
    pr_number=63,
    limit=5,
)
print(len(rows))
print(rows[0] if rows else "no rows")

E0000 00:00:1762434748.820428 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762434748.823934 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
2025-11-06 18:42:29,077 [INFO] Loading DATABASE_URL and configuring SQLAlchemy engine
E0000 00:00:1762434749.229835 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762434749.231280 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


1
{'id': '771dbbbe-15a6-4662-8601-4ca822ca807f', 'org_id': '76d33fb3-6062-456b-a211-4aec9971f8be', 'installation_id': '3afe9067-a516-494a-96fe-6038e4315557', 'repository_id': '33c22b77-9f63-454a-8a40-72713757be8f', 'repo_full_name': 'Intellytics-Solutions/github-app-poc', 'pr_number': 63, 'pr_title': 'Test 1', 'analysis_data': {'files': [{'sql_change': 'File: test_change.sql (added) [+1/-0]\n@@ -0,0 +1 @@\n+ALTER TABLE transform_zone.edw.gl_summary DROP COLUMN deptcd;', 'impact_analysis': '📌 **Change Summary**\n\nChange summary description: _Dropping the `deptcd` column from the `gl_summary` table will impact any downstream tables that depend on this column. This includes tables that use `deptcd` for calculations, filtering, or joining._\n\n| Field                  | Description |\n|------------------------|-------------|\n| Change Type            | Drop Column |\n| Affected Table         | transform_zone.edw.gl_summary |\n| Affected Column(s)     | deptcd |\n| Requested Change       |

In [ ]:
from langchain.agents import initialize_agent, AgentType
from app.tools import build_org_pr_repo_tool, LLM

org_id = "76d33fb3-6062-456b-a211-4aec9971f8be"
pr_repo_tool = build_org_pr_repo_tool(org_id=org_id, default_limit=10)

agent = initialize_agent(
    tools=[pr_repo_tool],
    llm=LLM,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

resp = agent.invoke("Show PR 63 in Intellytics-Solutions/github-app-poc")
print(resp)

E0000 00:00:1762494427.725620 7957248 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762494427.728268 7957248 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
2025-11-07 11:17:07,908 [INFO] Loading DATABASE_URL and configuring SQLAlchemy engine
E0000 00:00:1762494428.043462 7957248 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762494428.044483 7957248 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_37798/1517051530.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and



> Entering new AgentExecutor chain...
I need to retrieve the analysis for PR 63 in the Intellytics-Solutions/github-app-poc repository.
Action: pr_repo_analysis
Action Input: Intellytics-Solutions/github-app-poc PR 63

2025-11-07 11:17:18,737 [WARNING] Failed to get installation token for 93243242: Could not parse the provided public key.



Observation: Found 1 stored PR analysis record(s).
1. Intellytics-Solutions/github-app-poc PR #63 - Test 1
   Impact Preview: 📌 **Change Summary**  Change summary description: _Dropping the `deptcd` column from the `gl_summary` table will impact any downstream tables that depend on this column. This includes tables that use `deptcd` for calculations, filtering, or joining._  | Field                  | Description | |------------------------|-------------|

DATA:
{"analyses": [{"id": "771dbbbe-15a6-4662-8601-4ca822ca807f", "org_id": "76d33fb3-6062-456b-a211-4aec9971f8be", "installation_id": "3afe9067-a516-494a-96fe-6038e4315557", "repository_id": "33c22b77-9f63-454a-8a40-72713757be8f", "repo_full_name": "Intellytics-Solutions/github-app-poc", "pr_number": 63, "pr_title": "Test 1", "analysis_data": {"files": [{"sql_change": "File: test_change.sql (added) [+1/-0]\n@@ -0,0 +1 @@\n+ALTER TABLE transform_zone.edw.gl_summary DROP COLUMN deptcd;", "impact_analysis": "\ud83d\udccc **Change Sum

KeyboardInterrupt: 

In [5]:
from app.tools import fetch_pr_files_with_contents

resp = fetch_pr_files_with_contents(
    org_id="76d33fb3-6062-456b-a211-4aec9971f8be",
    repo_full_name="Intellytics-Solutions/github-app-poc",
    pr_number=63,
    which_ref="head",
)
print(resp)

2025-11-10 10:44:51,121 [INFO] fetch_pr_files_with_contents: org_id=76d33fb3-6062-456b-a211-4aec9971f8be, repo=Intellytics-Solutions/github-app-poc, pr_number=63, which_ref=head
2025-11-10 10:44:55,990 [INFO] Using ref=test-1 (sha=9f24fd671e985ea4efe6229e4677d4750441f3d4) for which_ref=head
2025-11-10 10:45:00,697 [INFO] Fetched 1 changed files for PR #63 in Intellytics-Solutions/github-app-poc
2025-11-10 10:45:00,698 [INFO] Found 1 changed files in PR
2025-11-10 10:45:04,607 [INFO] Completed: fetched contents for 1/1 files


{'ref': 'test-1', 'ref_sha': '9f24fd671e985ea4efe6229e4677d4750441f3d4', 'files': [{'filename': 'test_change.sql', 'status': 'added', 'additions': 1, 'deletions': 0, 'patch': '@@ -0,0 +1 @@\n+ALTER TABLE transform_zone.edw.gl_summary DROP COLUMN deptcd;', 'raw_url': 'https://github.com/Intellytics-Solutions/github-app-poc/raw/9f24fd671e985ea4efe6229e4677d4750441f3d4/test_change.sql', 'contents_url': 'https://api.github.com/repos/Intellytics-Solutions/github-app-poc/contents/test_change.sql?ref=9f24fd671e985ea4efe6229e4677d4750441f3d4', 'content': 'ALTER TABLE transform_zone.edw.gl_summary DROP COLUMN deptcd;\n'}]}


In [1]:
from langchain.agents import initialize_agent, AgentType
from app.tools import build_org_code_suggestion_tool, LLM, CODE_SUGGESTION_LLM

# Print which LLM models are being used
print("=" * 80)
print("LLM Configuration:")
print("=" * 80)

# Get agent LLM info
if LLM:
    agent_model = "unknown"
    if hasattr(LLM, 'model_name'):
        agent_model = LLM.model_name
    elif hasattr(LLM, 'model'):
        agent_model = LLM.model
    elif hasattr(LLM, '_model_name'):
        agent_model = LLM._model_name
    llm_class = type(LLM).__name__
    print(f"Agent LLM: {llm_class} - Model: {agent_model}")
else:
    print("Agent LLM: Not configured")

# Get code suggestion LLM info
if CODE_SUGGESTION_LLM:
    code_model = "unknown"
    if hasattr(CODE_SUGGESTION_LLM, 'model_name'):
        code_model = CODE_SUGGESTION_LLM.model_name
    elif hasattr(CODE_SUGGESTION_LLM, 'model'):
        code_model = CODE_SUGGESTION_LLM.model
    elif hasattr(CODE_SUGGESTION_LLM, '_model_name'):
        code_model = CODE_SUGGESTION_LLM._model_name
    code_class = type(CODE_SUGGESTION_LLM).__name__
    print(f"Code Suggestion LLM: {code_class} - Model: {code_model}")
else:
    print("Code Suggestion LLM: Not configured (will fallback to Agent LLM)")
print("=" * 80)
print()

org_id = "340e9d70-9ff3-4c64-9d39-5d25a4c5b85c"
code_suggestion_tool = build_org_code_suggestion_tool(org_id=org_id)

agent = initialize_agent(
    tools=[code_suggestion_tool],
    llm=LLM,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

# Test the code suggestion tool
resp = agent.invoke("I am removing the deptcd column in transform_zone.edw.gl_summary table, can you suggest the code changes required for impacted tables?")
print(resp)

E0000 00:00:1764223712.625216 14587492 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1764223712.627423 14587492 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
2025-11-27 11:38:33,000 [INFO] Loading DATABASE_URL and configuring SQLAlchemy engine
E0000 00:00:1764223713.123214 14587492 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1764223713.124404 14587492 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_32039/1661332781.py:42: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state

LLM Configuration:
Agent LLM: ChatGoogleGenerativeAI - Model: models/gemini-2.0-flash
Code Suggestion LLM: ChatOpenAI - Model: gpt-5-mini



> Entering new AgentExecutor chain...


2025-11-27 11:38:35,256 [INFO] Detected user-described changes: I am removing the deptcd column in transform_zone.edw.gl_summary table....
2025-11-27 11:38:35,257 [INFO] Analyzing described changes for org 340e9d70-9ff3-4c64-9d39-5d25a4c5b85c: I am removing the deptcd column in transform_zone.edw.gl_summary table....
/Users/deepkatbamna/Desktop/query_guard/QueryGuard-Backend/app/services/impact_analysis.py:108: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(
2025-11-27 11:38:35,426 [INFO] Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Okay, I need to identify the code changes required in tables impacted by the removal of the `deptcd` column from the `transform_zone.edw.gl_summary` table. I can use the `code_suggestion` tool to get specific code suggestions based on this change.
Action: code_suggestion
Action Input: I am removing the deptcd column in transform_zone.edw.gl_summary table.

/Users/deepkatbamna/Desktop/query_guard/QueryGuard-Backend/app/services/impact_analysis.py:541: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(q)
2025-11-27 11:38:36,130 [INFO] lineage.retrieval - org=340e9d70-9ff3-4c64-9d39-5d25a4c5b85c query_len=71 docs=8
2025-11-27 11:38:36,130 [INFO] lineage.retrieval.preview[0]: source_database: transform_zone source_schema: edw source_table: gl_summary source_column: deptcd target_database: transform_zone target_schema: edw target_table: allocationrule target_column: departmentlist query_id: 01c023d5-3202-0e04-0000-0010fc8a5a19 query_type: standalone_insert dependency_scor
2025-11-27 11:38:36,130 [INFO] lineage.retrieval.preview[1]: source_database: transform_zone source_schema: edw source_table: gl_summary source_column: accountcd target_database: transform_zone target_schema: 


Observation: ## Code Suggestions for Described Changes

**Change Description:** I am removing the deptcd column in transform_zone.edw.gl_summary table.


### File: unknown

❌ Error: Failed to generate suggestions: Error code: 404 - {'error': {'message': 'Your organization must be verified to use the model `gpt-5-mini`. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


⚠️ **CRITICAL:** No actual SQL code changes were provided!
The suggestions above are generic descriptions. Please ensure the LLM
returns actual SQL code in the 'code_changes' array with 'old_code' and 'new_code' fields.


---
*Structured data available in response*

Thought:

2025-11-27 11:38:59,151 [INFO] Detected user-described changes: What are the general considerations when removing a column from a table that is used in other tables...
2025-11-27 11:38:59,153 [INFO] Analyzing described changes for org 340e9d70-9ff3-4c64-9d39-5d25a4c5b85c: What are the general considerations when removing a column from a table that is used in other tables...


The code suggestion tool failed to provide specific SQL code changes due to an error. It seems like the model `gpt-5-mini` is not accessible because the organization needs to be verified. I need to try again with a different model or a more generic request. Since I don't have control over the model being used, I will rephrase the request to be more general and hope that it triggers a simpler response that doesn't rely on the unavailable model.

Action: code_suggestion
Action Input: What are the general considerations when removing a column from a table that is used in other tables or views?

2025-11-27 11:38:59,583 [INFO] lineage.retrieval - org=340e9d70-9ff3-4c64-9d39-5d25a4c5b85c query_len=110 docs=8
2025-11-27 11:38:59,584 [INFO] lineage.retrieval.preview[0]: target_database: prod_tz target_schema: edw_staging target_table: dimcharge target_column: updatedatetime query_id: ['01c06249-3202-1b64-0010-fc8a0004178e', '01c0624a-3202-1b20-0010-fc8a0004083a'] query_type: create_view_and_merge dependency_score: 1
2025-11-27 11:38:59,584 [INFO] lineage.retrieval.preview[1]: source_database: prod_tz source_schema: edw_staging source_table: chargecodemap source_column: chargename_consolidated target_database: prod_tz target_schema: edw_staging target_table: dimcharge target_column: chargename query_id: ['01c06249-3202-1b64-0010-fc8a0004178e', '01c0624a-3202-1b20-0010-fc8
2025-11-27 11:38:59,585 [INFO] lineage.retrieval.preview[2]: source_database: prod_tz source_schema: edw_staging source_table: chargecodemap source_column: rowsourcesystem target_database: prod_tz target_schema: e


Observation: ## Code Suggestions for Described Changes

**Change Description:** What are the general considerations when removing a column from a table that is used in other tables or views?


### File: unknown

❌ Error: Failed to generate suggestions: Error code: 404 - {'error': {'message': 'Your organization must be verified to use the model `gpt-5-mini`. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


⚠️ **CRITICAL:** No actual SQL code changes were provided!
The suggestions above are generic descriptions. Please ensure the LLM
returns actual SQL code in the 'code_changes' array with 'old_code' and 'new_code' fields.


---
*Structured data available in response*

Thought:The code suggestion tool continues to fail due to the same error. It appears I cannot get specific cod

## Jira tool

In [1]:
from app.tools import create_jira_ticket_for_org
import json

org_id = "76d33fb3-6062-456b-a211-4aec9971f8be"

result = create_jira_ticket_for_org(
    org_id=org_id,
    summary="Fix issue for impacted tables",
    description="Downstream table is broken for GL Amount → GL Amount Raw. Need to update process definition.",
    issue_type="Task",
    priority="High",
    assignee="deep.katbamna@intellyticssolutions.com",
    pr_url="https://github.com/org/repo/pull/123",
    analysis_report_url="https://zane.ai/reports/analysis-456",
)

print("Jira Ticket Creation Result:")
print(json.dumps(result, indent=2))

E0000 00:00:1762754080.674885 8500211 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762754080.680059 8500211 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
2025-11-10 11:24:40,849 [INFO] Loading DATABASE_URL and configuring SQLAlchemy engine
E0000 00:00:1762754080.954148 8500211 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762754080.955094 8500211 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
2025-11-10 11:24:41,009 [INFO] Creating Jira ticket for org_id=76d33fb3-6062-456b-a211-4aec9971f8be, summary=Fix issue for impacted tables...
2025-11-10 11:24:46,060 [INFO] Created Jira issue: QUEGAI-29
2025-11-10 11:24:48,781 [ERROR] Failed to save ticket to database: null value in column "id" of relation "jira_tickets" violates not-null constraint
DETAIL:  Failing r

Jira Ticket Creation Result:
{
  "success": true,
  "ticket": null,
  "jira_issue": {
    "key": "QUEGAI-29",
    "id": "10462",
    "url": "https://intellyticssolutions.atlassian.net/browse/QUEGAI-29"
  },
  "warning": "Ticket created in Jira but database save failed: null value in column \"id\" of relation \"jira_tickets\" violates not-null constraint\nDETAIL:  Failing row contains (null, 0a23aa65-ae6d-4e57-9726-41d2eeaf0cd5, QUEGAI-29, https://intellyticssolutions.atlassian.net/browse/QUEGAI-29, Fix issue for impacted tables, Downstream table is broken for GL Amount \u2192 GL Amount Raw. Need..., Task, Open, High, deep.katbamna@intellyticssolutions.com, https://github.com/org/repo/pull/123, https://zane.ai/reports/analysis-456, 0e9b500a-9999-4c56-9cb1-9841c10e1c22, 2025-11-10 05:54:48.073405+00, null).\n"
}
